<!-- nav -->
[← 5 · Operators](05_Operators.ipynb) · [Home](../../README.md) · [7 · Bindings →](07_Bindings.ipynb)

# 6 · Calls, members and subscripts

Tutorial 5 built expressions from operators. Most code also reaches into values: it reads a property (`x.celsius`), an
item (`x[0]`) or calls a function (`f(x)`). Most languages write these as postfix operations on a value, and each
dialect models them with kinds of its own, which this notebook builds in the builders' calls, beside each language's
text. Where two languages mean different things by the same text, or one has no counterpart for another's operation,
the translators say so rather than guess.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `variable` writes a
name in the dialects whose names are the framework's `variable` (tutorial 4), as tutorial 5 does, and `translate`
translates an expression from one dialect to another, as case study 14 does. Where a model is Python's, Python also
reads its text, with `Python3.Text.FromText`, and checks that it gives the same term (tutorial 3).

In [1]:
from mbse.Expressions import Translators
from mbse.Expressions.Patterns.Expressions import Evaluators, Expressions as E
from mbse.Expressions.Python import Python3
from mbse.Expressions.TypeScript import TypeScript5
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog
from mbse.Expressions.Excel import Expressions as Excel
from mbse.Expressions.MATLAB import Expressions as Matlab
from mbse.Expressions.Framework import Terms
from toolkit import dialects, show

store = dialects()
S, Term = store.Expressions, store.Expressions.Framework.Term
amd64 = store.Instances.Expressions.SystemV.AMD64
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))
languages = {"Python3": (Python3.Expressions.DIALECT, Python3.Text), "C": (C.Expressions.DIALECT, C.Text),
             "SystemVerilog": (SystemVerilog.Expressions.DIALECT, SystemVerilog.Text),
             "Excel": (Excel.Expressions.DIALECT, Excel.Text), "MATLAB": (Matlab.Expressions.DIALECT, Matlab.Text)}


def translate(expression, source, target):
    """`expression`, of the dialect `source`, translated to `target`, written as a term."""
    with store.transaction():
        return Term(Translators.between(source, target).forward(expression))

<a id="T06.1"></a>
## 06.1 · A property

Patterns reads a property with an operation, `get`: `get(x, 'celsius')` is a `call` whose `callable` is `get`, its
arguments the value and the property's name, a literal. Each language has a kind of its own for `x.celsius`: Python3's
`attribute` (`ast`'s `Attribute`, its `attr` a string), TypeScript5's `memberexpression` (ESTree's, its `property` an
`identifier` term), C's `memberexpression` (its `operator` `.` or `->`, its `member` a name), SystemVerilog's
`memberexpression` (its `member` an identifier), and Excel's and MATLAB's `field` (its `name` a string):

In [2]:
with store.transaction():
    get = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("get").arguments([
        lambda t: t.patterns(variable("x")),
        lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar("celsius"))))]))))
    members = {
        "Python3": (Python3.Text, S.Python.Python3.Expression(lambda e: e.root(
            lambda u: u.attribute(lambda a: a.value(variable("x")).attr("celsius"))))),
        "TypeScript5": (TypeScript5.Text, S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.memberexpression(lambda m: m
                .object(lambda o: o.identifier(lambda i: i.name("x")))
                .property(lambda p: p.identifier(lambda i: i.name("celsius"))))))),
        "C": (C.Text, S.C.Expressions.Expression(lambda e: e.root(
            lambda u: u.memberexpression(lambda m: m
                .object(variable("x")).operator(".").member(lambda n: n.identifier(lambda i: i.spelling("celsius"))))
        ).abi([amd64]))),
        "C, through a pointer": (C.Text, S.C.Expressions.Expression(lambda e: e.root(
            lambda u: u.memberexpression(lambda m: m
                .object(variable("x")).operator("->").member(lambda n: n.identifier(lambda i: i.spelling("celsius"))))
        ).abi([amd64]))),
        "SystemVerilog": (SystemVerilog.Text, S.SystemVerilog.Expressions.Expression(lambda e: e.root(
            lambda u: u.memberexpression(lambda m: m
                .value(lambda v: v.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x")))))
                .member(lambda i: i.spelling("celsius")))))),
        "Excel": (Excel.Text, S.Excel.Expressions.Expression(lambda e: e.root(
            lambda u: u.field(lambda f: f.value(variable("x")).name("celsius"))))),
        "MATLAB": (Matlab.Text, S.MATLAB.Expressions.Expression(lambda e: e.root(
            lambda u: u.field(lambda f: f.value(variable("x")).name("celsius"))))),
    }
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("x.celsius")))
print(f"{'Patterns':21}", show(get))
for language, (text, member) in members.items():
    print(f"{language:21}", f"{text.ToText(member):12}", Terms.form(Terms.term(member)).kind)
assert [text.ToText(member) for text, member in members.values()] == [
    "x.celsius", "x.celsius", "x.celsius", "x->celsius", "x.celsius", "=x.celsius", "x.celsius"]
assert Terms.same(read, members["Python3"][1])  # Python reads its own text into the same term

Patterns              get(x, 'celsius')
Python3               x.celsius    attribute
TypeScript5           x.celsius    memberexpression
C                     x.celsius    memberexpression
C, through a pointer  x->celsius   memberexpression
SystemVerilog         x.celsius    memberexpression
Excel                 =x.celsius   field
MATLAB                x.celsius    field


The translators write `get` as each language's own member kind, the same terms as built above (TypeScript5 has no
translator yet). An object's related objects, `entries(x, 'loggers')` (case study 11), are Patterns' alone: they are
read through the store's relations, which no language's member reaches, and every translator says so:

In [3]:
with store.transaction():
    entries = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("entries")
        .arguments([lambda t: t.patterns(variable("x")),
                    lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar("loggers"))))]))))
for language, (dialect, text) in languages.items():
    translated = translate(get, E.DIALECT, dialect)
    assert Terms.same(translated, members[language][1])
    try:
        print(f"{language:14}", f"{text.ToText(translated):12}", text.ToText(translate(entries, E.DIALECT, dialect)))
    except ValueError as error:
        print(f"{language:14}", f"{text.ToText(translated):12}", error)

Python3        x.celsius    Patterns call 'entries' has no Python3 counterpart
C              x.celsius    Patterns call 'entries' has no C counterpart
SystemVerilog  x.celsius    Patterns call 'entries' has no SystemVerilog counterpart
Excel          =x.celsius   Patterns call 'entries' has no Excel counterpart
MATLAB         x.celsius    Patterns call 'entries' has no MATLAB counterpart


<a id="T06.2"></a>
## 06.2 · An item

Patterns' `item(x, 0)` is a collection's first item: it counts from 0, as Python, TypeScript, C and SystemVerilog do.
Each writes `x[0]`, each with a kind of its own: Python3's `subscript`, TypeScript5's `memberexpression` with
`computed` set (in JavaScript, `x[0]` and `x.celsius` are one operation), C's `subscriptexpression` and SystemVerilog's
`indexexpression`. Excel and MATLAB count from 1: Excel has no subscript, but a function, `INDEX`; MATLAB's `index` is
`x(i)`, written as a call is. So their item 0 is `INDEX(x, 0 + 1)` and `x(0 + 1)`:

In [4]:
with store.transaction():
    item = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("item").arguments([
        lambda t: t.patterns(variable("x")),
        lambda t: t.patterns(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(0))))]))))
    items = {
        "Python3": (Python3.Text, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.subscript(lambda s: s
            .value(variable("x")).slice(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(0)).text("0").base(10))))))),
        "TypeScript5": (TypeScript5.Text, S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.memberexpression(lambda m: m
                .object(lambda o: o.identifier(lambda i: i.name("x"))).computed(True)
                .property(lambda p: p.literal(lambda r: r.value(lambda v: v.as_scalar(0.0)).text("0"))))))),
        "C": (C.Text, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.subscriptexpression(lambda s: s
            .object(variable("x")).indices([lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(0)))]))).abi([amd64]))),
        "SystemVerilog": (SystemVerilog.Text, S.SystemVerilog.Expressions.Expression(lambda e: e.root(
            lambda u: u.indexexpression(lambda s: s
                .value(lambda v: v.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x")))))
                .index(lambda d: d.literal(lambda r: r.value(lambda v: v.as_scalar(0)))))))),
        "Excel": (Excel.Text, S.Excel.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
            .callable("INDEX").arguments([
                lambda t: t.excel(variable("x")),
                lambda t: t.excel(lambda d: d.infix(lambda i: i
                    .left(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar(0)))).operator("+")
                    .right(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar(1))))))]))))),
        "MATLAB": (Matlab.Text, S.MATLAB.Expressions.Expression(lambda e: e.root(lambda u: u.index(lambda s: s
            .value(variable("x")).index(lambda d: d.binary(lambda b: b
                .left(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar(0)))).operator("+")
                .right(lambda l: l.literal(lambda r: r.value(lambda v: v.as_scalar(1)))))))))),
    }
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("x[0]")))
print(f"{'Patterns':14}", show(item), "of [3.5, 4.5] is", Evaluators.OfAny(item, {"x": [3.5, 4.5]}))
for language, (text, built) in items.items():
    print(f"{language:14}", f"{text.ToText(built):17}", Terms.form(Terms.term(built)).kind)
assert [text.ToText(built) for text, built in items.values()] == [
    "x[0]", "x[0]", "x[0]", "x[0]", "=INDEX(x, 0 + 1)", "x(0 + 1)"]
assert Terms.same(read, items["Python3"][1])

Patterns       item(x, 0) of [3.5, 4.5] is 3.5
Python3        x[0]              subscript
TypeScript5    x[0]              memberexpression
C              x[0]              subscriptexpression
SystemVerilog  x[0]              indexexpression
Excel          =INDEX(x, 0 + 1)  call
MATLAB         x(0 + 1)          index


Translating `item` writes out each language's counting, so that item 0 is the same item everywhere. C has a subscript,
but over arrays and pointers, which hold no length; translating Patterns' collections to C is a capability still to
come ([CAP-102: collections translated to LaTeX and C and C++](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-102.md)),
and its translator refuses `item` by name rather than write a subscript that means something else:

In [5]:
for language, (dialect, text) in languages.items():
    try:
        translated = translate(item, E.DIALECT, dialect)
        print(f"{language:14}", text.ToText(translated))
        assert text.ToText(translated) == text.ToText(items[language][1])
    except ValueError as error:
        print(f"{language:14}", error)

Python3        x[0]
C              Patterns call 'item' has no C counterpart
SystemVerilog  x[0]
Excel          =INDEX(x, 0 + 1)
MATLAB         x(0 + 1)


<a id="T06.3"></a>
## 06.3 · A call

Patterns calls an operation by name: in `max(x)` the `callable` is an attribute, the string `max`, never a value, and
validation refuses a name its dialect does not define (tutorial 5, 05.6). Excel and MATLAB call functions the same way,
with the framework's `call`, so `=MAX(x)` and `max(x)` read back as Patterns' `max(x)`:

In [6]:
with store.transaction():
    biggest = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("max")
        .arguments([lambda t: t.patterns(variable("x"))]))))
    unknown = S.Patterns.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("f")
        .arguments([lambda t: t.patterns(variable("x"))]))))
    excel = S.Excel.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("MAX")
        .arguments([lambda t: t.excel(variable("x"))]))))
    matlab = S.MATLAB.Expressions.Expression(lambda e: e.root(lambda u: u.call(lambda c: c.callable("max")
        .arguments([lambda t: t.matlab(variable("x"))]))))
form = Terms.form(Terms.term(biggest))
print(form.kind, form.attributes["callable"], len(form.arguments))
print(*E.DIALECT.validate(unknown, {"x"}))
for language, text, built in [("Excel", Excel.Text, excel), ("MATLAB", Matlab.Text, matlab)]:
    dialect = languages[language][0]
    print(f"{language:7}", f"{text.ToText(built):8}", Terms.form(Terms.term(built)).kind, show(translate(built, dialect, E.DIALECT)))

call max 1
'f' is not an operation Patterns defines
Excel   =MAX(x)  call max(x)
MATLAB  max(x)   call max(x)


Python3's `call` applies any term, its `func`: a name, an attribute, another call's result. So do TypeScript5's
`callexpression` (its `callee`) and C's (its `function`). Python's `max(x)` names a builtin that Patterns has, and reads
back as Patterns' `max(x)`; `f(x)` calls a value, which Patterns cannot, and its translator says so:

In [7]:
with store.transaction():
    calls = {
        "Python3": (Python3.Text, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
            .func(variable("f")).args([variable("x")]))))),
        "Python3 builtin": (Python3.Text, S.Python.Python3.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
            .func(variable("max")).args([variable("x")]))))),
        "TypeScript5": (TypeScript5.Text, S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.callexpression(lambda c: c
                .callee(lambda o: o.identifier(lambda i: i.name("f")))
                .arguments([lambda o: o.identifier(lambda i: i.name("x"))]))))),
        "C": (C.Text, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.callexpression(lambda c: c
            .function(variable("f")).arguments([variable("x")]))).abi([amd64]))),
    }
    read = [S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText(source))) for source in ("f(x)", "max(x)")]
for language, (text, built) in calls.items():
    form = Terms.form(Terms.term(built))
    print(f"{language:16}", f"{text.ToText(built):7}", form.kind, len(form.arguments))
assert [Terms.same(r, calls[name][1]) for r, name in zip(read, ("Python3", "Python3 builtin"))] == [True, True]
python = languages["Python3"][0]
print(show(translate(calls["Python3 builtin"][1], python, E.DIALECT)))
try:
    translate(calls["Python3"][1], python, E.DIALECT)
except ValueError as error:
    print(error)

Python3          f(x)    call 2
Python3 builtin  max(x)  call 2
TypeScript5      f(x)    callexpression 2
C                f(x)    callexpression 2
max(x)
Python3 call has no Patterns counterpart


SystemVerilog calls a system function by name, `$clog2(x)`, a `systemcall`, and an array's methods as calls of a member,
`x.sum()`, a `callexpression` whose `callee` is a `memberexpression`. A method that Patterns has reads back as its
operation; a system function has no counterpart, and its translator says so:

In [8]:
with store.transaction():
    system = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.systemcall(lambda c: c
        .name("$clog2").arguments([
            lambda t: t.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x"))))]))))
    method = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.callexpression(lambda c: c
        .callee(lambda t: t.memberexpression(lambda m: m
            .value(lambda v: v.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x")))))
            .member(lambda i: i.spelling("sum"))))
        .arguments([]))))
for built in (system, method):
    try:
        back = show(translate(built, SystemVerilog.Expressions.DIALECT, E.DIALECT))
    except ValueError as error:
        back = str(error)
    print(f"{SystemVerilog.Text.ToText(built):9}", f"{Terms.form(Terms.term(built)).kind:14}", back)

$clog2(x) systemcall     SystemVerilog systemcall has no Patterns counterpart
x.sum()   callexpression sum(x)


<a id="T06.4"></a>
## 06.4 · A chain

Postfix operations nest left to right: in `a.b.c(d)[e:f]` the subscript applies to the call's result, the call to the
attribute `c`, and `c` to the attribute `b` of `a`. So the outermost term is the last operation written, and the first
name written, `a`, is the deepest. `Terms.walk` reads the tree parents first; a slice, `e:f`, is a term of its own
(Python3's `slice`, its `lower`, `upper` and `step`), the subscript's `slice`:

In [9]:
with store.transaction():
    chain = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.subscript(lambda s: s
        .value(lambda v: v.call(lambda c: c
            .func(lambda f: f.attribute(lambda a: a
                .value(lambda b: b.attribute(lambda a: a.value(variable("a")).attr("b"))).attr("c")))
            .args([variable("d")])))
        .slice(lambda d: d.slice(lambda r: r.lower(variable("e")).upper(variable("f")))))))
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("a.b.c(d)[e:f]")))
print(Python3.Text.ToText(chain))
assert Python3.Text.ToText(chain) == "a.b.c(d)[e:f]" and Terms.same(read, chain)
for term in Terms.walk(chain):
    form = Terms.form(term)
    print(f"{form.kind:10}", ", ".join(f"{name}={value}" for name, value in form.attributes.items()))

a.b.c(d)[e:f]
subscript  
call       
attribute  attr=c
attribute  attr=b
variable   name=a
variable   name=d
slice      
variable   name=e
variable   name=f


<a id="T06.5"></a>
## 06.5 · Helpers are parts, not terms

Python's `f(x, y=1)` passes `y=1` by keyword. A keyword is not a term: `y=1` is no expression of its own, and nothing
evaluates it. It is a helper, Python3's `keyword` (`ast`'s), a value object a call holds in its `keywords`, its `arg` a
name and its `value` a term. A form reads a helper as a `Terms.Part` among its term's attributes: its kind, its own
attributes, and how many of the term's arguments are its terms, which follow the term's own subterms and variadic
arguments. So the call's arguments are `f`, `x`, then the keyword's value, `1`, and `make` writes the form back:

In [10]:
with store.transaction():
    keyword = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.call(lambda c: c
        .func(variable("f")).args([variable("x")])
        .keywords([lambda k: k.arg("y").value(lambda d: d.literal(
            lambda r: r.value(lambda v: v.as_scalar(1)).text("1").base(10)))]))))
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("f(x, y=1)")))
    made = Term(Python3.Expressions.DIALECT.make(Terms.form(Terms.term(keyword))))
form = Terms.form(Terms.term(keyword))
print(Python3.Text.ToText(keyword))
print(form.kind, *form.attributes["keywords"])
print(*(Python3.Text.ToText(argument) for argument in form.arguments), sep=", ")
assert form.attributes["keywords"] == (Terms.Part("keyword", {"arg": "y"}, 1),)
assert Terms.same(read, keyword) and Terms.same(made, keyword)

f(x, y=1)
call Part(kind='keyword', attributes={'arg': 'y'}, size=1)
f, x, 1


## What you learned

- **Each language reaches into values its own way.** A property is Python3's `attribute`, TypeScript5's, C's and
  SystemVerilog's `memberexpression`, Excel's and MATLAB's `field`; Patterns reads it with its operation `get`.
- **An item's index is the language's.** Patterns, Python, TypeScript, C and SystemVerilog count from 0; Excel and
  MATLAB from 1, and the translators write the difference out.
- **A call names an operation, or applies a value.** Patterns, Excel and MATLAB call by name; Python3, TypeScript5, C
  and SystemVerilog apply a term. What has no counterpart, a call of a value in Patterns, Patterns' `item` in C,
  Patterns' `entries` anywhere else, is refused by name, never guessed.
- **Postfix operations nest left to right**, the last written outermost.
- **A helper is a part, not a term**: a form keeps its structure in its attributes and its terms in its arguments.

Next, tutorial 7 binds names: `let`, lambdas, quantifiers and comprehensions.

<!-- nav -->
[← 5 · Operators](05_Operators.ipynb) · [Home](../../README.md) · [7 · Bindings →](07_Bindings.ipynb)